# Eksperyment 2: Wplyw rozmiaru okna cech dynamicznych na jakosc predykcji

## Cel eksperymentu
Zbadanie, jak rozmiar **sliding window** (liczba ostatnich meczow branych pod uwage przy obliczaniu formy gracza) wplywa na jakosc predykcji modelu.

## Hipoteza
- **Zbyt male okno** (np. 3 mecze) — duza wariancja formy, szum dominuje nad sygnalem
- **Zbyt duze okno** (np. 50 meczow) — forma jest wygladzona, ale nie reaguje na biezace zmiany
- **Optymalne okno** — rownowazymy szum z aktualnoscią informacji

## Metodologia
Testujemy okna: **3, 5, 10, 15, 20, 30** meczow. Dla kazdego okna:
1. Przeliczamy cechy dynamiczne (forma, forma nawierzchniowa)
2. Trenujemy Random Forest z tymi samymi hiperparametrami
3. Ewaluujemy na zbiorze testowym (Accuracy, Log Loss, Brier Score)

**Uwaga:** H2H zawsze korzysta z pelnej historii (nie ma sensu go okienkowac).

In [1]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
WINDOW_SIZES = [3, 5, 10, 15, 20, 30]

## 1. Przygotowanie danych bazowych

Wczytanie i podzial danych — identycznie jak w modelu bazowym. Cechy dynamiczne zostana obliczone osobno dla kazdego rozmiaru okna.

In [2]:
# --- Wczytanie danych ---
df = pd.read_csv('sample_data/2024.csv')
df['tourney_date'] = pd.to_datetime(df['tourney_date'], format='%Y%m%d')
df = df.sort_values(['tourney_date', 'match_num']).reset_index(drop=True)

cols_base = ['surface', 'tourney_level', 'winner_rank', 'winner_age', 'winner_ht',
             'loser_rank', 'loser_age', 'loser_ht', 'winner_name', 'loser_name']
df_base = df[cols_base].dropna().copy()
df_base['winner_rank_log'] = np.log(df_base['winner_rank'])
df_base['loser_rank_log'] = np.log(df_base['loser_rank'])

# --- Dane historyczne ---
history_parts = []
for filepath in ['sample_data/2022.csv', 'sample_data/2023.csv']:
    df_hist = pd.read_csv(filepath)
    df_hist['tourney_date'] = pd.to_datetime(df_hist['tourney_date'], format='%Y%m%d')
    df_hist = df_hist.sort_values(['tourney_date', 'match_num']).reset_index(drop=True)
    history_parts.append(df_hist[cols_base].dropna().copy())
df_history_base = pd.concat(history_parts, ignore_index=True)

# --- Label Encoding ---
le_surface = LabelEncoder()
le_level = LabelEncoder()
le_surface.fit(pd.concat([df_base['surface'], df_history_base['surface']]).unique())
le_level.fit(pd.concat([df_base['tourney_level'], df_history_base['tourney_level']]).unique())
df_base['surface_encoded'] = le_surface.transform(df_base['surface'])
df_base['tourney_level_encoded'] = le_level.transform(df_base['tourney_level'])

# --- Podzial chronologiczny ---
train_end = int(len(df_base) * 0.60)
val_end = int(len(df_base) * 0.80)
df_train_raw_base = df_base.iloc[:train_end].reset_index(drop=True)
df_val_raw_base = df_base.iloc[train_end:val_end].reset_index(drop=True)
df_test_raw_base = df_base.iloc[val_end:].reset_index(drop=True)
df_train_raw_base['match_id'] = range(len(df_train_raw_base))
df_val_raw_base['match_id'] = range(len(df_val_raw_base))
df_test_raw_base['match_id'] = range(len(df_test_raw_base))

print(f"Dane 2024: {len(df_base)} meczow")
print(f"Historia: {len(df_history_base)} meczow")
print(f"Podzial: train={len(df_train_raw_base)}, val={len(df_val_raw_base)}, test={len(df_test_raw_base)}")
print(f"\nTestowane okna: {WINDOW_SIZES}")

Dane 2024: 3005 meczow
Historia: 5789 meczow
Podzial: train=1803, val=601, test=601

Testowane okna: [3, 5, 10, 15, 20, 30]


## 2. Funkcje cech dynamicznych z parametrycznym oknem

Funkcje `calculate_form` i `calculate_surface_form` przyjmuja teraz parametr `window_size` zamiast stalej wartosci 10.

In [3]:
def calculate_form(player_name, history, window_size=10):
    """Win rate z ostatnich window_size meczow."""
    ph = history[(history['winner_name'] == player_name) |
                 (history['loser_name'] == player_name)].tail(window_size)
    if len(ph) == 0:
        return 0.5
    return len(ph[ph['winner_name'] == player_name]) / len(ph)

def get_h2h(p1, p2, history):
    """Bilans H2H — zawsze pelna historia (bez okienkowania)."""
    return (len(history[(history['winner_name'] == p1) & (history['loser_name'] == p2)]) -
            len(history[(history['winner_name'] == p2) & (history['loser_name'] == p1)]))

def calculate_surface_form(player_name, surface, history, window_size=10):
    """Forma na nawierzchni z ostatnich window_size meczow na danej nawierzchni."""
    sm = history[history['surface'] == surface]
    ps = sm[(sm['winner_name'] == player_name) | (sm['loser_name'] == player_name)].tail(window_size)
    if len(ps) < 3:
        return calculate_form(player_name, history, window_size)
    return len(ps[ps['winner_name'] == player_name]) / len(ps)

def add_dynamic_features(df_subset, historical_data, window_size=10):
    """Cechy dynamiczne z expanding window i zadanym rozmiarem okna formy."""
    h2h_list, w_form_list, l_form_list, w_sf_list, l_sf_list = [], [], [], [], []
    full_sequence = pd.concat([historical_data, df_subset]).reset_index(drop=True)
    start_idx = len(historical_data)
    for i in range(len(df_subset)):
        row = df_subset.iloc[i]
        past = full_sequence.iloc[:start_idx + i]
        h2h_list.append(get_h2h(row['winner_name'], row['loser_name'], past))
        w_form_list.append(calculate_form(row['winner_name'], past, window_size))
        l_form_list.append(calculate_form(row['loser_name'], past, window_size))
        w_sf_list.append(calculate_surface_form(row['winner_name'], row['surface'], past, window_size))
        l_sf_list.append(calculate_surface_form(row['loser_name'], row['surface'], past, window_size))
    df_subset = df_subset.copy()
    df_subset['h2h_diff'] = h2h_list
    df_subset['w_form'] = w_form_list
    df_subset['l_form'] = l_form_list
    df_subset['w_surface_form'] = w_sf_list
    df_subset['l_surface_form'] = l_sf_list
    return df_subset

def symmetrize_data(df_subset, shuffle=True):
    rows_p1_wins, rows_p2_wins = [], []
    for idx, row in df_subset.iterrows():
        base = {'match_id': row['match_id'], 'surface': row['surface_encoded'],
                'tourney_level': row['tourney_level_encoded']}
        row1 = {**base,
                'p1_rank_log': row['winner_rank_log'], 'p1_age': row['winner_age'], 'p1_ht': row['winner_ht'],
                'p2_rank_log': row['loser_rank_log'], 'p2_age': row['loser_age'], 'p2_ht': row['loser_ht'],
                'p1_h2h': row['h2h_diff'], 'p1_form': row['w_form'], 'p2_form': row['l_form'],
                'p1_surface_form': row['w_surface_form'], 'p2_surface_form': row['l_surface_form'],
                'rank_diff': row['winner_rank_log'] - row['loser_rank_log'],
                'age_diff': row['winner_age'] - row['loser_age'],
                'ht_diff': row['winner_ht'] - row['loser_ht'],
                'form_diff': row['w_form'] - row['l_form'],
                'y': 1, 'actual_winner': row['winner_name'], 'actual_loser': row['loser_name'],
                'p1_name': row['winner_name'], 'p2_name': row['loser_name']}
        row2 = {**base,
                'p1_rank_log': row['loser_rank_log'], 'p1_age': row['loser_age'], 'p1_ht': row['loser_ht'],
                'p2_rank_log': row['winner_rank_log'], 'p2_age': row['winner_age'], 'p2_ht': row['winner_ht'],
                'p1_h2h': -row['h2h_diff'], 'p1_form': row['l_form'], 'p2_form': row['w_form'],
                'p1_surface_form': row['l_surface_form'], 'p2_surface_form': row['w_surface_form'],
                'rank_diff': row['loser_rank_log'] - row['winner_rank_log'],
                'age_diff': row['loser_age'] - row['winner_age'],
                'ht_diff': row['loser_ht'] - row['winner_ht'],
                'form_diff': row['l_form'] - row['w_form'],
                'y': 0, 'actual_winner': row['winner_name'], 'actual_loser': row['loser_name'],
                'p1_name': row['loser_name'], 'p2_name': row['winner_name']}
        rows_p1_wins.append(row1)
        rows_p2_wins.append(row2)
    all_rows = [r for pair in zip(rows_p1_wins, rows_p2_wins) for r in pair]
    result = pd.DataFrame(all_rows)
    if shuffle:
        result = result.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return result

features = ['surface', 'tourney_level', 'p1_rank_log', 'p1_age', 'p1_ht',
            'p2_rank_log', 'p2_age', 'p2_ht', 'p1_h2h', 'p1_form', 'p2_form',
            'p1_surface_form', 'p2_surface_form',
            'rank_diff', 'age_diff', 'ht_diff', 'form_diff']

print("Funkcje gotowe.")

Funkcje gotowe.


## 3. Petla eksperymentalna

Dla kazdego rozmiaru okna przeliczamy cechy dynamiczne od zera i trenujemy **ten sam model** (Random Forest ze stalymi hiperparametrami), aby jedyna zmieniana zmienna bylo okno formy.

Uzywamy **stalych hiperparametrow** (a nie ponownej optymalizacji), aby izolowac wplyw rozmiaru okna od losowosci wyszukiwania hiperparametrow.

In [4]:
# Stale hiperparametry RF (zblizone do optymalnych z modelu bazowego)
RF_PARAMS = {
    'n_estimators': 300,
    'max_depth': 20,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'bootstrap': True,
    'max_samples': 0.8,
    'random_state': RANDOM_STATE,
    'n_jobs': -1
}

experiment_results = []

for w in WINDOW_SIZES:
    print(f"\n{'='*50}")
    print(f"Okno = {w} meczow")
    print('='*50)

    # Kopiujemy dane bazowe (bez cech dynamicznych)
    df_train = df_train_raw_base.copy()
    df_val = df_val_raw_base.copy()
    df_test = df_test_raw_base.copy()

    # Obliczanie cech dynamicznych z nowym oknem
    df_train = add_dynamic_features(df_train, df_history_base, window_size=w)
    hist_val = pd.concat([df_history_base, df_train[cols_base]]).reset_index(drop=True)
    df_val = add_dynamic_features(df_val, hist_val, window_size=w)
    hist_test = pd.concat([df_history_base, df_train[cols_base], df_val[cols_base]]).reset_index(drop=True)
    df_test = add_dynamic_features(df_test, hist_test, window_size=w)

    # Symetryzacja
    train_sym = symmetrize_data(df_train, shuffle=True)
    test_sym = symmetrize_data(df_test, shuffle=True)

    X_train = train_sym[features]
    y_train = train_sym['y']
    X_test = test_sym[features]
    y_test = test_sym['y']

    # Trening
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_train, y_train)

    # Ewaluacja
    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba)
    brier = brier_score_loss(y_test, y_proba)

    # Match-level
    test_sym_copy = test_sym.copy()
    test_sym_copy['p1_win_prob'] = y_proba
    wp = test_sym_copy[test_sym_copy['y'] == 1]
    match_acc = (wp['p1_win_prob'] > 0.5).mean()
    match_brier = ((1.0 - wp['p1_win_prob']) ** 2).mean()

    result = {
        'Window': w,
        'Accuracy': acc,
        'Match Accuracy': match_acc,
        'Log Loss': ll,
        'Brier Score': brier,
        'Match Brier': match_brier
    }
    experiment_results.append(result)

    print(f"  Accuracy:       {acc:.4f}")
    print(f"  Match Accuracy: {match_acc:.4f}")
    print(f"  Log Loss:       {ll:.4f}")
    print(f"  Brier Score:    {brier:.4f}")

print("\nEksperyment zakonczony.")


Okno = 3 meczow
  Accuracy:       0.6240
  Match Accuracy: 0.6140
  Log Loss:       0.6524
  Brier Score:    0.2307

Okno = 5 meczow
  Accuracy:       0.5932
  Match Accuracy: 0.6040
  Log Loss:       0.6632
  Brier Score:    0.2359

Okno = 10 meczow
  Accuracy:       0.6065
  Match Accuracy: 0.6106
  Log Loss:       0.6666
  Brier Score:    0.2367

Okno = 15 meczow
  Accuracy:       0.6015
  Match Accuracy: 0.5907
  Log Loss:       0.6578
  Brier Score:    0.2335

Okno = 20 meczow
  Accuracy:       0.6156
  Match Accuracy: 0.6206
  Log Loss:       0.6552
  Brier Score:    0.2320

Okno = 30 meczow
  Accuracy:       0.6065
  Match Accuracy: 0.6090
  Log Loss:       0.6515
  Brier Score:    0.2302

Eksperyment zakonczony.


## 4. Tabela wynikow

In [5]:
df_exp = pd.DataFrame(experiment_results).set_index('Window')
print(df_exp.round(4).to_string())

        Accuracy  Match Accuracy  Log Loss  Brier Score  Match Brier
Window                                                              
3         0.6240          0.6140    0.6524       0.2307       0.2324
5         0.5932          0.6040    0.6632       0.2359       0.2338
10        0.6065          0.6106    0.6666       0.2367       0.2369
15        0.6015          0.5907    0.6578       0.2335       0.2329
20        0.6156          0.6206    0.6552       0.2320       0.2303
30        0.6065          0.6090    0.6515       0.2302       0.2294


## 5. Analiza wynikow i wnioski z eksperymentu

In [6]:
print("=" * 60)
print("ANALIZA WPLYWU ROZMIARU OKNA")
print("=" * 60)
print()
print(df_exp[['Match Accuracy', 'Log Loss', 'Brier Score']].round(4).to_string())
print()

best_w_acc = df_exp['Match Accuracy'].idxmax()
best_w_ll = df_exp['Log Loss'].idxmin()
best_w_brier = df_exp['Brier Score'].idxmin()

print(f"Najlepsze okno wg Match Accuracy: {best_w_acc} meczow ({df_exp.loc[best_w_acc, 'Match Accuracy']:.4f})")
print(f"Najlepsze okno wg Log Loss:       {best_w_ll} meczow ({df_exp.loc[best_w_ll, 'Log Loss']:.4f})")
print(f"Najlepsze okno wg Brier Score:    {best_w_brier} meczow ({df_exp.loc[best_w_brier, 'Brier Score']:.4f})")

spread_acc = df_exp['Match Accuracy'].max() - df_exp['Match Accuracy'].min()
print(f"\nRozstep Match Accuracy: {spread_acc:.4f} ({spread_acc*100:.2f} p.p.)")
print(f"Rozstep Log Loss:       {df_exp['Log Loss'].max() - df_exp['Log Loss'].min():.4f}")

if spread_acc < 0.02:
    print("\nWniosek: Rozmiar okna ma NIEWIELKI wplyw na jakosc predykcji (<2 p.p.).")
    print("Model jest odporny na ten hiperparametr — ranking i inne cechy statyczne dominuja.")
elif spread_acc < 0.05:
    print("\nWniosek: Rozmiar okna ma UMIARKOWANY wplyw na jakosc predykcji (2-5 p.p.).")
    print(f"Zalecane okno: {best_w_acc} meczow.")
else:
    print("\nWniosek: Rozmiar okna ma ISTOTNY wplyw na jakosc predykcji (>5 p.p.).")
    print(f"Zalecane okno: {best_w_acc} meczow.")

ANALIZA WPLYWU ROZMIARU OKNA

        Match Accuracy  Log Loss  Brier Score
Window                                       
3               0.6140    0.6524       0.2307
5               0.6040    0.6632       0.2359
10              0.6106    0.6666       0.2367
15              0.5907    0.6578       0.2335
20              0.6206    0.6552       0.2320
30              0.6090    0.6515       0.2302

Najlepsze okno wg Match Accuracy: 20 meczow (0.6206)
Najlepsze okno wg Log Loss:       30 meczow (0.6515)
Najlepsze okno wg Brier Score:    30 meczow (0.2302)

Rozstep Match Accuracy: 0.0300 (3.00 p.p.)
Rozstep Log Loss:       0.0150

Wniosek: Rozmiar okna ma UMIARKOWANY wplyw na jakosc predykcji (2-5 p.p.).
Zalecane okno: 20 meczow.


## 6. Podsumowanie i wnioski

Wyniki eksperymentu pokazuja, ze rozmiar okna formy ma wprawdzie pewien wplyw na jakosc predykcji, ale nie jest to czynnik decydujacy. Rozstep miedzy najlepsza a najgorsza trafnoscia meczowa wynosi kilka punktow procentowych — to wplyw umiarkowany. Cechy statyczne (przede wszystkim ranking ATP) nadal odgrywaja wieksza role w przewidywaniu wyniku meczu niz sama wartosc formy.

Co ciekawe, nie ma jednego „idealnego" rozmiaru okna, ktory byłby najlepszy we wszystkich metrykach jednoczesnie. Wieksze okna (20-30 meczow) daja na ogol lepiej skalibrowane prawdopodobienstwa (nizszy Log Loss i Brier Score), co ma sens — forma obliczona z dluzszej historii jest bardziej stabilna i mniej podatna na szum wynikajacy z kilku przypadkowych wynikow. Z kolei pod wzgledem samej trafnosci binarnej (Match Accuracy) wyniki sa mniej jednoznaczne i zaleza od konkretnego zbioru testowego.

Warto zwrocic uwage na pewien nieintuicyjny rezultat: bardzo male okno (3 mecze) wcale nie wypada najgorzej. Mozna to wytlumaczyc tym, ze krotkie okno dobrze reaguje na nagle zmiany formy gracza — np. gdy zawodnik wchodzi w serie zwyciestw albo wraca po kontuzji. Srednie okna (5-15 meczow) moga natomiast „rozmywac" te krotkoterminowe trendy, mieszajac stare i nowe wyniki.

Jesli chodzi o weryfikacje hipotez postawionych na poczatku eksperymentu — potwierdzily sie one czesciowo. Zbyt male okno rzeczywiscie wprowadza wieksza wariancje, choc nie przekłada sie to na najgorsze wyniki. Zbyt duze okno wygladza forme kosztem aktualnosci, ale poprawia kalibracje. Optymalne okno istnieje, ale jego wartosc zalezy od tego, na jakiej metryce nam najbardziej zalezy.

Bilans H2H (head-to-head) celowo nie podlegal okienkowaniu — zawodnicy graja ze soba na tyle rzadko, ze ograniczanie historii wzajemnych spotkan nie mialoby uzasadnienia.

Na podstawie wynikow mozna zarekomendowac stosowanie okna w zakresie 20 meczow jako rozsadnego kompromisu miedzy reaktywnoscia na zmiany formy a stabilnoscia obliczen. Warto jednak pamietac, ze roznice sa niewielkie i w praktyce wazniejsze moze byc wzbogacenie modelu o dodatkowe cechy niz precyzyjne dobranie tego jednego parametru.